## Prueba del Modelo Exportado — Clasificación de Diabetes
Este notebook carga `mejor_modelo_diabetes.pkl` (generado por el **Bloque 10** de `ProyectoFinal.ipynb`) y lo aplica sobre **data nueva**.

**Requisitos de la data nueva (CSV):**
- Debe tener las 8 columnas: `Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age`
- La columna `Outcome` es **opcional**: si está presente, además de predecir se evalúa el modelo.

> Ejecutar primero `ProyectoFinal.ipynb` completo para que exista el `.pkl`.

In [ ]:
# ── Cargar el artefacto exportado ─────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

art = joblib.load('mejor_modelo_diabetes.pkl')

print('=' * 60)
print('  ARTEFACTO CARGADO')
print('=' * 60)
print(f"  Modelo          : {art['modelo_nombre']}")
print(f"  ROC AUC (test)  : {art['metricas_test']['ROC AUC']:.4f}")
print(f"  F1-Score (test) : {art['metricas_test']['F1-Score']:.4f}")
print(f"  Exportado       : {art['fecha_export']}")
print(f"  Features        : {art['features']}")
print('=' * 60)

### 1 · Cargar la data nueva
Cambiar `RUTA_DATA_NUEVA` por la ruta de tu archivo CSV. Como ejemplo se usa `Testing.csv`.

In [ ]:
RUTA_DATA_NUEVA = 'Testing.csv'   # ← cambiar por tu archivo

df_nuevo = pd.read_csv(RUTA_DATA_NUEVA)
print(f'Data nueva: {df_nuevo.shape[0]} filas x {df_nuevo.shape[1]} columnas')
print(f"Incluye Outcome: {'SÍ' if 'Outcome' in df_nuevo.columns else 'NO'}")
df_nuevo.head()

### 2 · Preprocesamiento
Se aplica **exactamente el mismo pipeline del entrenamiento**, con los parámetros aprendidos en el train (nunca recalculados sobre la data nueva):

1. Ceros imposibles → NaN → **mediana del train**
2. Winsorizing con **límites IQR×3 del train**
3. `log1p` a las columnas sesgadas
4. Escalado con el **RobustScaler ajustado en train**

In [ ]:
def preprocesar(df, art):
    """Aplica a la data nueva el mismo preprocesamiento del entrenamiento."""
    df = df.copy()

    faltantes = [c for c in art['features'] if c not in df.columns]
    if faltantes:
        raise ValueError(f'Faltan columnas en la data nueva: {faltantes}')

    # 1. Ceros imposibles → NaN → mediana del train
    for col in art['cols_ceros']:
        df[col] = df[col].replace(0, np.nan)
        df[col] = df[col].fillna(art['medianas_imputacion'][col])

    # 2. Winsorizing con límites del train
    for col, (lo, hi) in art['limites_iqr'].items():
        df[col] = df[col].clip(lower=lo, upper=hi)

    # 3. log1p a columnas sesgadas
    for col in art['cols_log1p']:
        df[col] = np.log1p(df[col])

    # 4. Escalado con el scaler del train
    X = pd.DataFrame(art['scaler'].transform(df[art['features']]),
                     columns=art['features'])
    return X

X_nuevo = preprocesar(df_nuevo, art)
print('Preprocesamiento aplicado:')
X_nuevo.head()

### 3 · Predicciones
Se genera la probabilidad de diabetes y la clase predicha para cada registro. El resultado se guarda en `predicciones.csv`.

In [ ]:
modelo = art['modelo']

proba = modelo.predict_proba(X_nuevo)[:, 1]
pred  = modelo.predict(X_nuevo)

resultados = df_nuevo.copy()
resultados['Prob_Diabetes'] = proba.round(4)
resultados['Prediccion']    = pred
resultados['Riesgo']        = pd.cut(proba, bins=[0, 0.3, 0.6, 1.0],
                                     labels=['Bajo', 'Medio', 'Alto'])

print(f'Positivos predichos: {int(pred.sum())} de {len(pred)} ({pred.mean():.1%})')
display(resultados.sort_values('Prob_Diabetes', ascending=False).head(10))

resultados.to_csv('predicciones.csv', index=False)
print('\n✅ Resultados guardados en predicciones.csv')

# Distribución de probabilidades
plt.figure(figsize=(8, 4))
plt.hist(proba, bins=30, color='steelblue', alpha=0.8, edgecolor='white')
plt.axvline(0.5, color='red', linestyle='--', label='Umbral 0.5')
plt.title(f'Distribución de Probabilidades — {art["modelo_nombre"]}',
          fontsize=12, fontweight='bold')
plt.xlabel('Probabilidad de Diabetes')
plt.ylabel('Cantidad de registros')
plt.legend()
plt.tight_layout()
plt.show()

### 4 · Evaluación (solo si la data nueva incluye `Outcome`)
Si el CSV trae la clase real, se calculan las métricas y se grafica la matriz de confusión y la curva ROC.

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             ConfusionMatrixDisplay, roc_curve, cohen_kappa_score)

if 'Outcome' in df_nuevo.columns:
    y_real = df_nuevo['Outcome']

    print('=' * 50)
    print(f"  EVALUACIÓN — {art['modelo_nombre']}")
    print('=' * 50)
    print(f"  Accuracy  : {accuracy_score(y_real, pred):.4f}")
    print(f"  Precision : {precision_score(y_real, pred, zero_division=0):.4f}")
    print(f"  Recall    : {recall_score(y_real, pred, zero_division=0):.4f}")
    print(f"  F1-Score  : {f1_score(y_real, pred, zero_division=0):.4f}")
    print(f"  Kappa     : {cohen_kappa_score(y_real, pred):.4f}")
    print(f"  ROC AUC   : {roc_auc_score(y_real, proba):.4f}")
    print(f"  Gini      : {2*roc_auc_score(y_real, proba) - 1:.4f}")
    print('=' * 50)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    cm = confusion_matrix(y_real, pred)
    ConfusionMatrixDisplay(cm, display_labels=['Sin Diab.', 'Con Diab.']).plot(
        ax=axes[0], colorbar=False, cmap='Blues')
    axes[0].set_title('Matriz de Confusión', fontsize=11, fontweight='bold')

    fpr, tpr, _ = roc_curve(y_real, proba)
    axes[1].plot(fpr, tpr, color='steelblue', linewidth=2,
                 label=f'AUC = {roc_auc_score(y_real, proba):.4f}')
    axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1)
    axes[1].set_xlabel('Tasa de Falsos Positivos (FPR)')
    axes[1].set_ylabel('Tasa de Verdaderos Positivos (TPR)')
    axes[1].set_title('Curva ROC', fontsize=11, fontweight='bold')
    axes[1].legend(loc='lower right')

    plt.suptitle(f'Evaluación con Data Nueva — {art["modelo_nombre"]}',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('La data nueva NO incluye Outcome → solo se generaron predicciones (ver predicciones.csv).')